# 02 — Find Emotion Features

Goal: per emotion, rank SAE features by `mean(emotion) - mean(neutral)`. Save `features.json`.

Assumes Notebook 01 ran (model + SAE work).

In [1]:
import json
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sae_lens import SAE

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
MODEL_ID = 'google/gemma-3-1b-it'
LAYER = 13
SAE_ID = f'layer_{LAYER}_width_16k_l0_medium'

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map=device).eval()
sae, _, _ = SAE.from_pretrained(release='gemma-scope-2-1b-it-res', sae_id=SAE_ID, device=device)
print('ready')

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

ready


/var/folders/k2/qttynbv17pj1sbr20x6pcsc80000gn/T/ipykernel_89233/680308018.py:16: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, _, _ = SAE.from_pretrained(release='gemma-scope-2-1b-it-res', sae_id=SAE_ID, device=device)


## Prompt sets

Short emotional declarations. Bias toward first-person + present tense — concentrates feature signal.

In [2]:
EMOTIONS = {
    'anger': [
        'I am absolutely furious right now.',
        'This makes me so angry I can barely speak.',
        'I want to scream at the top of my lungs.',
        'My blood is boiling with rage.',
        'I have never been this enraged in my life.',
        'I am livid and ready to break something.',
        'How dare they do this to me, I am seething.',
        'My fists are clenched and my teeth are grinding.',
        'I am so mad I could punch a wall.',
        'Rage courses through every part of my body.',
        'I despise this and everyone involved.',
        'I am incensed by what just happened.',
    ],
    'sadness': [
        'I am so heartbroken I can barely breathe.',
        'Tears stream down my face without stopping.',
        'I feel a heavy emptiness in my chest.',
        'Everything reminds me of what I lost.',
        'I am drowning in sorrow today.',
        'My grief feels like it will never end.',
        'I miss them so much it physically hurts.',
        'I cannot stop crying no matter what I do.',
        'A deep sadness has settled into my bones.',
        'I feel completely alone and broken inside.',
        'Despair is the only word for how I feel.',
        'I am mourning everything I will never have.',
    ],
    'joy': [
        'I am so incredibly happy right now!',
        'My heart is bursting with pure delight.',
        'I cannot stop smiling, this is wonderful.',
        'Today is the best day of my entire life.',
        'I feel like I am floating on a cloud.',
        'Pure joy fills every cell of my body.',
        'I am over the moon with happiness.',
        'Laughter keeps bubbling up from inside me.',
        'I am beaming and I want to hug everyone.',
        'This is the most wonderful thing ever.',
        'I am giddy with excitement and joy.',
        'My heart sings with absolute happiness.',
    ],
    'fear': [
        'I am terrified and shaking uncontrollably.',
        'My heart is pounding out of my chest in fear.',
        'I am afraid of what is in the dark.',
        'Cold dread washes over me.',
        'I am too scared to even move.',
        'Panic grips me and I cannot think clearly.',
        'I am petrified by what might happen next.',
        'A chill of pure terror runs down my spine.',
        'I am trembling with fright.',
        'My hands shake from sheer terror.',
        'I dread what is coming and I cannot escape.',
        'I have never been so frightened in my life.',
    ],
    'disgust': [
        'That is absolutely revolting and disgusting.',
        'I want to vomit just thinking about it.',
        'The smell makes me gag in horror.',
        'I am repulsed beyond words.',
        'This is the most vile thing I have ever seen.',
        'I cannot stomach this any longer.',
        'Every part of me recoils in disgust.',
        'This is sickening and grotesque.',
        'I feel nauseous and need to look away.',
        'That is foul and putrid beyond belief.',
        'My skin crawls from how gross this is.',
        'I am revolted by what I just witnessed.',
    ],
    'surprise': [
        'I cannot believe what I am seeing right now.',
        'This is the most shocking thing ever.',
        'My jaw literally dropped to the floor.',
        'I am completely stunned and speechless.',
        'What on earth, I did not see that coming.',
        'I am floored, this is unbelievable.',
        'My eyes are wide with astonishment.',
        'I gasped out loud in shock.',
        'This caught me totally off guard.',
        'I am dumbfounded by this turn of events.',
        'Wow, I am genuinely amazed right now.',
        'I never expected this in a million years.',
    ],
    'love': [
        'I love you more than words can describe.',
        'My heart is so full of love for you.',
        'I adore everything about this person.',
        'I am head over heels and deeply in love.',
        'You mean absolutely everything to me.',
        'My love for you grows stronger every day.',
        'I cherish every moment we share together.',
        'I am so deeply in love it hurts.',
        'You are the love of my entire life.',
        'My heart belongs to you forever.',
        'I adore you with every fiber of my being.',
        'Loving you is the easiest thing I have ever done.',
    ],
    'anxiety': [
        'My chest is tight and I cannot calm down.',
        'I am spiraling with worry about everything.',
        'My thoughts are racing and I feel on edge.',
        'I am anxious and cannot sit still.',
        'A knot of worry sits in my stomach.',
        'I keep checking and rechecking everything nervously.',
        'My hands are clammy and my heart races.',
        'I am consumed by anxious dread.',
        'I cannot stop overthinking every detail.',
        'Restless worry keeps me awake all night.',
        'My mind will not stop catastrophizing.',
        'I feel jittery and unable to focus.',
    ],
}

NEUTRAL = [
    'The capital of France is Paris.',
    'Water boils at one hundred degrees Celsius.',
    'The library opens at nine in the morning.',
    'I went to the store to buy bread.',
    'The train arrives at platform three.',
    'Photosynthesis converts sunlight into chemical energy.',
    'The meeting is scheduled for Thursday afternoon.',
    'A triangle has three sides and three angles.',
    'The package will arrive next Tuesday.',
    'He picked up the book and turned the page.',
    'Coffee is grown in many tropical countries.',
    'The road continues for another five miles.',
    'My laptop has sixteen gigabytes of memory.',
    'The recipe calls for two cups of flour.',
    'Birds typically migrate south for the winter.',
    'The conference room is on the second floor.',
    'She closed the window because it was raining.',
    'Mathematics is taught in schools worldwide.',
    'The bus stops at the corner every ten minutes.',
    'I read the article in this morning.',
]
print({e: len(v) for e, v in EMOTIONS.items()}, 'neutral:', len(NEUTRAL))

{'anger': 12, 'sadness': 12, 'joy': 12, 'fear': 12, 'disgust': 12, 'surprise': 12, 'love': 12, 'anxiety': 12} neutral: 20


## Feature extractor

Hook layer 13, batch encode prompts, get mean SAE activation per prompt (over non-pad tokens).

In [3]:
@torch.no_grad()
def feat_activations(prompts, batch_size=8):
    """Return [n_prompts, d_sae] mean SAE feature activation per prompt (non-pad tokens)."""
    cache = {}
    def hook(_, __, output):
        cache['h'] = (output[0] if isinstance(output, tuple) else output).detach()
    handle = model.model.layers[LAYER].register_forward_hook(hook)
    out = []
    try:
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i:i+batch_size]
            enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=64).to(device)
            _ = model(**enc)
            h = cache['h'].to(torch.float32)              # [B, T, d_model]
            feats = sae.encode(h)                          # [B, T, d_sae]
            mask = enc['attention_mask'].unsqueeze(-1).float()  # [B, T, 1]
            per_prompt = (feats * mask).sum(1) / mask.sum(1).clamp(min=1)  # [B, d_sae]
            out.append(per_prompt.cpu())
    finally:
        handle.remove()
    return torch.cat(out, 0)

neutral_acts = feat_activations(NEUTRAL)
print('neutral acts:', neutral_acts.shape)

neutral acts: torch.Size([20, 16384])


In [4]:
emo_acts = {e: feat_activations(p) for e, p in EMOTIONS.items()}
for e, a in emo_acts.items():
    print(f'{e:10s} {tuple(a.shape)}')

anger      (12, 16384)
sadness    (12, 16384)
joy        (12, 16384)
fear       (12, 16384)
disgust    (12, 16384)
surprise   (12, 16384)
love       (12, 16384)
anxiety    (12, 16384)


## Rank features per emotion

Score = mean(emotion) - mean(neutral). Top-k by score.

Also report consistency: fraction of emotion prompts where feature fires above neutral max — guards against one-prompt outliers.

In [5]:
TOP_K = 15
neutral_mean = neutral_acts.mean(0)
neutral_max = neutral_acts.max(0).values

results = {}
for emo, acts in emo_acts.items():
    emo_mean = acts.mean(0)
    diff = emo_mean - neutral_mean
    top_idx = torch.topk(diff, TOP_K).indices.tolist()
    consistency = ((acts > neutral_max.unsqueeze(0)).float().mean(0))
    rows = []
    for idx in top_idx:
        rows.append({
            'feat_id': int(idx),
            'diff_score': float(diff[idx]),
            'emo_mean_act': float(emo_mean[idx]),
            'neutral_mean_act': float(neutral_mean[idx]),
            'consistency': float(consistency[idx]),  # 0-1, frac prompts above neutral max
        })
    results[emo] = rows

for emo, rows in results.items():
    print(f'\n=== {emo} ===')
    for r in rows[:8]:
        print(f'  feat {r["feat_id"]:>5d}  diff={r["diff_score"]:.3f}  consistency={r["consistency"]:.2f}')


=== anger ===
  feat    92  diff=55.386  consistency=0.67
  feat   549  diff=46.717  consistency=0.83
  feat  5923  diff=44.960  consistency=0.83
  feat   544  diff=37.897  consistency=0.17
  feat   909  diff=37.751  consistency=0.92
  feat  2213  diff=35.255  consistency=0.58
  feat  2239  diff=33.429  consistency=1.00
  feat  5088  diff=33.252  consistency=1.00

=== sadness ===
  feat  2697  diff=58.914  consistency=1.00
  feat   909  diff=55.542  consistency=1.00
  feat   701  diff=45.226  consistency=0.83
  feat    92  diff=40.414  consistency=0.50
  feat   514  diff=38.498  consistency=1.00
  feat  2356  diff=36.957  consistency=0.75
  feat   159  diff=33.270  consistency=0.58
  feat   397  diff=31.359  consistency=0.33

=== joy ===
  feat    92  diff=58.412  consistency=0.58
  feat  2562  diff=57.639  consistency=1.00
  feat   549  diff=54.558  consistency=0.75
  feat  2356  diff=43.313  consistency=1.00
  feat   494  diff=34.811  consistency=0.83
  feat  5923  diff=34.415  cons

## Neuronpedia links

Print URLs to inspect auto-interp labels. URL pattern for Gemma Scope 2 1B-IT res:

`https://www.neuronpedia.org/gemma-scope-2-1b-it/<layer>-gemmascope-res-16k/<feat_id>`

(May need adjustment after Neuronpedia indexes — open one and confirm.)

In [6]:
NP_BASE = f'https://www.neuronpedia.org/gemma-scope-2-1b-it/{LAYER}-gemmascope-res-16k'
for emo, rows in results.items():
    print(f'\n{emo}:')
    for r in rows[:5]:
        print(f'  {NP_BASE}/{r["feat_id"]}  (diff={r["diff_score"]:.2f})')


anger:
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/92  (diff=55.39)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/549  (diff=46.72)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/5923  (diff=44.96)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/544  (diff=37.90)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/909  (diff=37.75)

sadness:
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/2697  (diff=58.91)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/909  (diff=55.54)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/701  (diff=45.23)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/92  (diff=40.41)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/514  (diff=38.50)

joy:
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/92  

## Save

In [7]:
out_path = Path('../features.json')
payload = {
    'model': MODEL_ID,
    'sae_release': 'gemma-scope-2-1b-it-res',
    'sae_id': SAE_ID,
    'layer': LAYER,
    'features': results,
}
out_path.write_text(json.dumps(payload, indent=2))
print('wrote', out_path.resolve())

wrote /Users/aaronrockmenezes/Desktop/Projects/Mech Interp/gemma-emotion-steer/features.json


## Next

- Open top-3 Neuronpedia links per emotion. Confirm labels match (e.g., anger features should mention rage/conflict/hostility).
- Drop bad features manually if auto-interp says off-topic (e.g., punctuation feature that happens to fire on exclamations).
- Then `steer.py` — CLI patch hook using `sae.W_dec[feat_id]`.